In [1]:
import torch
from model import Model
from train import train
from graph_generation import random_graph
from algorithms import simulate_bfs, simulate_bf, compute_states

### Create the model

In [2]:
model = Model(['bf'], 1, 32, 1)
train(model, train_size=200, num_nodes=10, epochs=100, patience=20, save=False) 

Generating training data... Generated!
Start training process...
Epoch 1 | train loss = 1.03 (state=0.64, predec=0.28, term=1.27) | val loss = 0.41 (state=0.06, predec=0.19, term=1.25)
Epoch 2 | train loss = 0.34 (state=0.08, predec=0.17, term=0.88) | val loss = 0.25 (state=0.05, predec=0.17, term=0.61)
Epoch 3 | train loss = 0.29 (state=0.06, predec=0.15, term=0.77) | val loss = 0.24 (state=0.05, predec=0.14, term=0.58)
Epoch 4 | train loss = 0.27 (state=0.06, predec=0.13, term=0.73) | val loss = 0.26 (state=0.09, predec=0.13, term=0.55)
Epoch 5 | train loss = 0.25 (state=0.05, predec=0.12, term=0.71) | val loss = 0.21 (state=0.04, predec=0.12, term=0.57)
Epoch 6 | train loss = 0.25 (state=0.05, predec=0.11, term=0.71) | val loss = 0.20 (state=0.04, predec=0.11, term=0.52)
Epoch 7 | train loss = 0.24 (state=0.05, predec=0.10, term=0.70) | val loss = 0.23 (state=0.08, predec=0.10, term=0.51)
Epoch 8 | train loss = 0.23 (state=0.05, predec=0.09, term=0.69) | val loss = 0.16 (state=0.03,

In [13]:
graph, source = random_graph(num_nodes=6, weighted=True, weight_mn=0.2, weight_mx=2.0, self_loop=True), 1
x = [0 if i == source else graph.get_longest_path() + 1 for i in range(graph.num_nodes)]

h = torch.zeros(graph.num_nodes, model.hidden_dim)
x = torch.tensor(x).unsqueeze(1)
edges = graph.get_edge_tensors(h.device)

states, predecessors, max_dist, terminations = compute_states('bf', graph, source)

for i in range(graph.num_nodes):
    h = torch.zeros(graph.num_nodes, model.hidden_dim)
    y_pred, p_pred, h, t_pred = model('bf', edges, x, h)

    print(y_pred.squeeze())

    h = h.detach()
    x = y_pred.detach()

    #if torch.sigmoid(t_pred).item() > 0.5:
        #break

print(x)
print(states[-1])

tensor([0.6852, 0.4097, 0.5525, 1.4520, 3.9956, 0.7591],
       grad_fn=<SqueezeBackward0>)
tensor([0.6164, 0.5097, 0.5173, 0.9230, 1.9068, 0.5426],
       grad_fn=<SqueezeBackward0>)
tensor([0.7613, 0.5563, 0.5548, 0.6725, 1.5153, 0.7149],
       grad_fn=<SqueezeBackward0>)
tensor([0.7627, 0.6209, 0.5903, 0.7299, 1.4794, 0.6461],
       grad_fn=<SqueezeBackward0>)
tensor([0.8200, 0.6624, 0.6191, 0.7419, 1.4328, 0.6982],
       grad_fn=<SqueezeBackward0>)
tensor([0.8386, 0.6907, 0.6445, 0.7828, 1.4229, 0.7228],
       grad_fn=<SqueezeBackward0>)
tensor([[0.8386],
        [0.6907],
        [0.6445],
        [0.7828],
        [1.4229],
        [0.7228]])
[0.8737185839272044, 0.0, 0.4928804380468105, 1.0400000675250822, 2.465777928851389, 0.7397792513414438]


In [14]:
print(states)

[[3.9586583668981996, 0.0, 3.9586583668981996, 3.9586583668981996, 3.9586583668981996, 3.9586583668981996], [0.8737185839272044, 0.0, 0.4928804380468105, 1.5791815927236388, 3.9586583668981996, 0.7397792513414438], [0.8737185839272044, 0.0, 0.4928804380468105, 1.0400000675250822, 2.465777928851389, 0.7397792513414438]]


In [15]:
pred_predecessors = p_pred.argmax(dim=1).tolist()
true_predecessors = predecessors[-1]

def reconstruct_path(preds, node, source, n):
    path = [node]
    current = node
    while current != source and len(path) <= n:
        current = preds[current]
        path.append(current)
    return list(reversed(path))

for node in range(graph.num_nodes):
    if node == source:
        continue
    pred_path = reconstruct_path(pred_predecessors, node, source, graph.num_nodes)
    true_path = reconstruct_path(true_predecessors, node, source, graph.num_nodes)
    pred_str = " -> ".join(map(str, pred_path))
    true_str = " -> ".join(map(str, true_path))
    print(f"Node {node}: {pred_str}  (expected: {true_str})")

Node 0: 3 -> 3 -> 3 -> 3 -> 3 -> 3 -> 0  (expected: 1 -> 0)
Node 2: 2 -> 2 -> 2 -> 2 -> 2 -> 2 -> 2  (expected: 1 -> 2)
Node 3: 3 -> 3 -> 3 -> 3 -> 3 -> 3 -> 3  (expected: 1 -> 5 -> 3)
Node 4: 4 -> 4 -> 4 -> 4 -> 4 -> 4 -> 4  (expected: 1 -> 5 -> 4)
Node 5: 3 -> 3 -> 3 -> 3 -> 3 -> 3 -> 5  (expected: 1 -> 5)


In [16]:
pred_predecessors

[3, 2, 2, 3, 4, 3]